# Ejercicio: Web Scraping

## Objetivo de la práctica

El objetivo de este ejercicio es construir un web scraper que recoja datos de un website.

### Parte 0: Planificar
1. Identificar los datos que quieres obtener.
2. Elegir el sitio web objetivo.
3. Planificar la estructura del corpus.

**Nombre:** José Armando Sarango Cuenca.

## Parte 1: Entender el sitio web objetivo

- Analizar la estructura de la página web a ser analizada.
- Identificar los elementos HTML que contienen los datos bsuscados.

In [23]:
from bs4 import BeautifulSoup

file = '../content/rotisserie-chicken.html'
# Load the HTML file
with open(file, "r", encoding="utf-8") as file:
    html_content = file.read()

# Parse the HTML content with BeautifulSoup
soup = BeautifulSoup(html_content, "html.parser")

In [24]:
# Extracting the recipe title
title = soup.find("meta", {"property": "og:title"})["content"]
title

'Rotisserie Chicken'

In [25]:
ingredients_section = soup.find_all("li", class_="mm-recipes-structured-ingredients__list-item")
for ingredient in ingredients_section:
    print(ingredient.text.strip())

1 (3 pound) whole chicken
1 pinch salt
¼ cup butter, melted
1 tablespoon salt
1 tablespoon ground paprika
¼ tablespoon ground black pepper


## Parte 2: Obtener los datos deseados

* Buscar dentro del contenido HTML y extraer la información.

In [26]:
# Extracting the description
description = soup.find("meta", {"name": "description"})["content"]

# Extracting the ingredients
ingredients_section = soup.find_all("li", class_="mm-recipes-structured-ingredients__list-item")
ingredients = [ingredient.get_text().strip() for ingredient in ingredients_section]

# Extracting the instructions
instructions_section = soup.find_all("p", class_="comp mntl-sc-block mntl-sc-block-html")
instructions = [instruction.get_text().strip() for instruction in instructions_section]

# Extracting the nutrition information
nutrition_section = soup.find_all("span", class_="mm-recipes-nutrition-facts-label__nutrient-name mm-recipes-nutrition-facts-label__nutrient-name--has-postfix")
nutrition_facts = [fact.parent.get_text().strip().replace('\n', ' ') for fact in nutrition_section]

# Print the extracted information
print("Recipe Title:", title)
print("Description:", description)
print("Ingredients:")
for ingredient in ingredients:
    print("-", ingredient)
print("Instructions:")
for i, instruction in enumerate(instructions, 1):
    print(f"{i}. {instruction}")
print("Nutrition Facts:")
for fact in nutrition_facts:
    print("-", fact)


Recipe Title: Rotisserie Chicken
Description: Rotisserie chicken that's easy to cook on a gas grill and turns out moist and juicy with crispy skin. This is a simple recipe that our family loves.
Ingredients:
- 1 (3 pound) whole chicken
- 1 pinch salt
- ¼ cup butter, melted
- 1 tablespoon salt
- 1 tablespoon ground paprika
- ¼ tablespoon ground black pepper
Instructions:
1. Intimidated by the idea of making a rotisserie chicken at home? We're here to help. Get your grill and rotisserie attachment ready — you'll want to try this recipe ASAP.
2. Here's what you'll need to make rotisserie chicken at home:
3. · Whole Chicken: This recipe is meant for a whole 3-pound chicken. If your chicken is larger or smaller, you'll have to adjust the cooking time.· Butter: Butter keeps the chicken moist and juicy, while giving the seasonings something to stick to.· Seasonings: The rotisserie chicken is simply seasoned with salt, pepper, and paprika.
4. You'll find the full, step-by-step recipe below — b

## Parte 3: Obtener enlaces relacionados

---


* Encontrar links a otras recetas para completar el corpus

In [27]:
# Find all the links to other recipes
recipe_links = soup.find_all("a", href=True)

# Filter and print only the links that are likely to be recipes
recipe_urls = []
for link in recipe_links:
    href = link['href']
    if "recipe" in href:
        recipe_urls.append(href)

# Print the recipe URLs
print("Linked Recipes:")
for url in recipe_urls:
    print(url)

Linked Recipes:
https://www.allrecipes.com/authentication/login?regSource=3675&relativeRedirectUrl=%2Frecipe%2F93168%2Frotisserie-chicken%2F
/account/add-recipe
https://www.myrecipes.com/favorites
https://www.allrecipes.com/authentication/logout?relativeRedirectUrl=%2Frecipe%2F93168%2Frotisserie-chicken%2F
https://www.magazines.com/allrecipes-magazine.html?utm_source=allrecipes.com&utm_medium=owned&utm_campaign=i111arr1w2661
https://www.magazines.com/allrecipes-magazine.html
https://www.allrecipes.com/recipes/17562/dinner/
https://www.allrecipes.com/recipes/17057/everyday-cooking/more-meal-ideas/5-ingredients/main-dishes/
https://www.allrecipes.com/recipes/15436/everyday-cooking/one-pot-meals/
https://www.allrecipes.com/recipes/1947/everyday-cooking/quick-and-easy/
https://www.allrecipes.com/recipes/455/everyday-cooking/more-meal-ideas/30-minute-meals/
https://www.allrecipes.com/recipes/17889/everyday-cooking/family-friendly/family-dinners/
https://www.allrecipes.com/recipes/94/soups-s

### Traer solo os links relacionados a recetas

In [28]:
import requests

# Find all the links to other recipes
all_links = soup.find_all("a", href=True)

# Redes sociales y páginas de sistema a excluir
excluded_patterns = [
    'facebook.com', 'instagram.com', 'pinterest.com', 'tiktok.com',
    'youtube.com', 'flipboard.com', 'twitter.com',
    '/authentication/', '/account/', 'magazines.com', '/cook/'
]

recipe_urls = []
for link in all_links:
    href = link['href']

    # Filtro estricto:
    # 1. Debe contener '/recipe/' (patrón de AllRecipes para recetas individuales)
    # 2. No debe contener patrones de redes sociales o páginas de cuenta/revistas
    if "/recipe/" in href.lower():
        if not any(pattern in href.lower() for pattern in excluded_patterns):
            # Normalizar rutas relativas
            if href.startswith('/'):
                href = "https://www.allrecipes.com" + href

            if href not in recipe_urls:
                recipe_urls.append(href)

# Imprimir los enlaces filtrados
print(f"Se encontraron {len(recipe_urls)} enlaces de recetas:")
for url in recipe_urls:
    print(url)

Se encontraron 16 enlaces de recetas genuinas:
https://www.allrecipes.com/recipe/238575/cilantro-lime-grilled-chicken/
https://www.allrecipes.com/recipe/275062/buttermilk-barbecue-chicken/
https://www.allrecipes.com/recipe/274724/grilled-spatchcocked-chicken/
https://www.allrecipes.com/recipe/14531/beer-butt-chicken/
https://www.allrecipes.com/recipe/221093/good-frickin-paprika-chicken/
https://www.allrecipes.com/recipe/264278/miso-honey-chicken/
https://www.allrecipes.com/recipe/258659/rosemary-buttermilk-chicken/
https://www.allrecipes.com/recipe/222936/smoked-beer-butt-chicken/
https://www.allrecipes.com/recipe/228070/the-best-beer-can-chicken-ever/
https://www.allrecipes.com/recipe/214619/bbq-beer-can-chicken/
https://www.allrecipes.com/recipe/19944/drunk-chicken/
https://www.allrecipes.com/recipe/275044/grilled-chicken-under-a-brick/
https://www.allrecipes.com/recipe/281255/smoked-whole-chicken/
https://www.allrecipes.com/recipe/34957/easy-barbeque-chicken/
https://www.allrecipes.

## Parte 4: Hacer RAG con las recetas obtenidas
* Una vez que se ha construido el corpus, implementar y desplegar RAG para realizar búsquedas en el corpus

### Construccion del corpus:

In [33]:
import os
from bs4 import BeautifulSoup

def parse_recipe(html, fuente=""):
    """Extrae los datos de una receta (misma lógica de tu Parte 2)."""
    soup = BeautifulSoup(html, "html.parser")

    t = soup.find("meta", {"property": "og:title"})
    title = t["content"] if t else ""

    d = soup.find("meta", {"name": "description"})
    description = d["content"] if d else ""

    ing = soup.find_all("li", class_="mm-recipes-structured-ingredients__list-item")
    ingredients = [i.get_text().strip() for i in ing]

    ins = soup.find_all("p", class_="comp mntl-sc-block mntl-sc-block-html")
    instructions = [p.get_text().strip() for p in ins]

    return {
        "url": fuente,
        "title": title,
        "description": description,
        "ingredients": ingredients,
        "instructions": instructions,
    }
data = ["/content"]
archivos = []
for carpeta in data:
    if os.path.isdir(carpeta):
        for nombre in os.listdir(carpeta):
            if nombre.endswith(".html"):
                archivos.append(os.path.join(carpeta, nombre))

print("Archivos HTML encontrados:", len(archivos))

# Construir el corpus
corpus = []
for ruta in archivos:
    with open(ruta, "r", encoding="utf-8") as f:
        html = f.read()
    rec = parse_recipe(html, fuente=os.path.basename(ruta))
    if rec["title"] and rec["ingredients"]:
        corpus.append(rec)
        print("OK       ->", rec["title"])
    else:
        print("SIN DATOS->", os.path.basename(ruta))

print("\n Recetas en el corpus:", len(corpus))

Archivos HTML encontrados: 17
OK       -> Cilantro-Lime Grilled Chicken
OK       -> Best Beer Can Chicken
OK       -> Drunk Chicken
OK       -> Rotisserie Chicken
OK       -> Darn Good Chicken
OK       -> Good Frickin’ Paprika Chicken
OK       -> Beer Can Chicken
OK       -> Smoked Whole Chicken
OK       -> Miso Honey Chicken
OK       -> Grilled Spatchcocked Chicken
OK       -> The Best Beer Can Chicken Ever
OK       -> Beer Butt Chicken
OK       -> Rosemary Buttermilk Chicken
OK       -> Smoked Beer Butt Chicken
OK       -> Buttermilk Barbecue Chicken
OK       -> Easy Barbeque Chicken
OK       -> Grilled Chicken Under a Brick

 Recetas en el corpus: 17


Corpus en dataframe

In [49]:
import pandas as pd
df_corpus = pd.DataFrame(corpus)
df_corpus["n_ingredientes"] = df_corpus["ingredients"].apply(len)
df_corpus["n_instrucciones"] = df_corpus["instructions"].apply(len)
print("Corpus:", df_corpus.shape[0], "recetas,", df_corpus.shape[1], "columnas")
df_corpus[["title", "n_ingredientes", "n_instrucciones", "url"]]

Corpus: 17 recetas, 7 columnas


,title,n_ingredientes,n_instrucciones,url
0,Cilantro-Lime Grilled Chicken,5,8,238575_cilantro-lime-grilled-chicken.html
1,Best Beer Can Chicken,9,8,214619_bbq-beer-can-chicken.html
2,Drunk Chicken,6,6,19944_drunk-chicken.html
3,Rotisserie Chicken,6,23,rotisserie-chicken.html
4,Darn Good Chicken,4,3,8998_darn-good-chicken.html
5,Good Frickin’ Paprika Chicken,14,8,221093_good-frickin-paprika-chicken.html
6,Beer Can Chicken,8,6,214618_beer-can-chicken.html
7,Smoked Whole Chicken,9,7,281255_smoked-whole-chicken.html
8,Miso Honey Chicken,9,7,264278_miso-honey-chicken.html
9,Grilled Spatchcocked Chicken,8,9,274724_grilled-spatchcocked-chicken.html


In [50]:
df_corpus.iloc[0]

,0
url,238575_cilantro-lime-grilled-chicken.html
title,Cilantro-Lime Grilled Chicken
description,This cilantro-lime grilled chicken recipe star...
ingredients,"[½ cup chopped fresh cilantro, 4 limes, juice..."
instructions,"[Whisk cilantro, lime juice, garlic salt, and ..."
n_ingredientes,5
n_instrucciones,8


In [ ]:
%%capture
!pip install -q langchain langchain-community sentence-transformers faiss-cpu google-generativeai

4.1 Preparar el Corpus y Embeddings
Convertiremos la información extraída (título, descripción, ingredientes e instrucciones) en documentos procesables.

In [51]:
from langchain_community.vectorstores import FAISS
from langchain_core.documents import Document
from langchain_community.embeddings import HuggingFaceEmbeddings
documents = []
for _, fila in df_corpus.iterrows():
    t   = fila["title"]
    d   = fila["description"]
    ing = ", ".join(fila["ingredients"])
    ins = " ".join(fila["instructions"])

    # Por cada receta creamos 3 chunks
    documents.append(Document(
        page_content=f"Receta: {t}. {d}",
        metadata={"receta": t, "seccion": "info"}
    ))
    documents.append(Document(
        page_content=f"Ingredientes de '{t}': {ing}",
        metadata={"receta": t, "seccion": "ingredientes"}
    ))
    documents.append(Document(
        page_content=f"Instrucciones de '{t}': {ins}",
        metadata={"receta": t, "seccion": "instrucciones"}
    ))

print("Total de documentos (chunks):", len(documents))

# Inicializamos los embeddings
embeddings = HuggingFaceEmbeddings(model_name="all-MiniLM-L6-v2")

# Creamos la base de datos vectorial  con  el corpus
vector_db = FAISS.from_documents(documents, embeddings)

print("Base de datos vectorial creada con éxito con", len(df_corpus), "recetas.")

Total de documentos (chunks): 51


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Base de datos vectorial creada con éxito con 17 recetas.


In [52]:

import google.generativeai as genai
from google.colab import userdata

try:
    GOOGLE_API_KEY = userdata.get('ApiDeArmando')
    genai.configure(api_key=GOOGLE_API_KEY)
    model_name = 'gemini-2.5-flash'
    model = genai.GenerativeModel(model_name)
    print(f"Modelo {model_name} configurado correctamente.")
except Exception as e:
    print(f"Error: {e}. No esta configurada el GOOGLE_API_KEY.")

Modelo gemini-2.5-flash configurado correctamente.


In [53]:
def ask_recipe(question, k=3):
    # 1. Búsqueda de similitud (Retrieval)
    docs = vector_db.similarity_search(question, k=k)
    context = "\n".join([d.page_content for d in docs])

    # Recetas de las que se sacó el contexto
    recetas_usadas = []
    for d in docs:
        r = d.metadata.get("receta", "desconocida")
        if r not in recetas_usadas:
            recetas_usadas.append(r)

    # 2.Prompt
    prompt = (
        "Eres un asistente de cocina. Utiliza únicamente el siguiente contexto "
        "para responder la pregunta del usuario. Si la información no está en el "
        "contexto, di claramente que no lo sabes.\n\n"
        f"Contexto:\n{context}\n\n"
        f"Pregunta: {question}"
    )

    # 3. Generación
    response = model.generate_content(prompt)

    return {
        "pregunta": question,
        "respuesta": response.text,
        "recetas_fuente": recetas_usadas,
    }

In [54]:
preguntas = [
    "¿A qué temperatura interna debe llegar el pollo para estar listo?",
    "¿Cómo preparo un pollo con sabor a lima y cilantro?",
    "¿Qué recetas usan cerveza (beer)?",
    "¿Qué vino recomiendan para acompañar?",   # NO está en el corpus
    "¿Como preparo cecina?"
]

for q in preguntas:
    try:
        r = ask_recipe(q)
        print("Pregunta :", r["pregunta"])
        print("Respuesta:", r["respuesta"].strip())
        print("Fuente   :", ", ".join(r["recetas_fuente"]))
    except Exception as e:
        print("Error al generar respuesta:", e)
    print("-" * 70)

Pregunta : ¿A qué temperatura interna debe llegar el pollo para estar listo?
Respuesta: Según las instrucciones proporcionadas, el pollo debe alcanzar una temperatura interna de 165 grados F (74 grados C).
Fuente   : Grilled Chicken Under a Brick, Easy Barbeque Chicken, Cilantro-Lime Grilled Chicken
----------------------------------------------------------------------
Pregunta : ¿Cómo preparo un pollo con sabor a lima y cilantro?
Respuesta: Para preparar un pollo con sabor a lima y cilantro, sigue estos pasos:

1.  En un tazón grande de vidrio o cerámica, bate el cilantro, el jugo de lima, la sal de ajo y la pimienta negra.
2.  Agrega el pollo y mézclalo para cubrirlo de manera uniforme.
3.  Cubre el tazón con papel film y marina en el refrigerador de 30 minutos a toda la noche. (Si tienes prisa, no es necesario marinar; simplemente mezcla el pollo y ponlo en la parrilla).
4.  Precalienta una parrilla exterior a fuego medio-alto y engrasa ligeramente la rejilla.
5.  Retira el pollo de